# Probabilistic Pipeline (Bayesian Ridge)

This notebook mirrors the split and preprocessing workflow from `ml_models_pipelines.ipynb`, then tunes Bayesian Ridge and evaluates probabilistic metrics.


In [109]:
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
import joblib
from scipy.stats import norm
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import BayesianRidge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
warnings.filterwarnings("ignore")


In [110]:
BASE_DIR = Path.cwd()
if not (BASE_DIR / "data" / "raw").exists():
    BASE_DIR = BASE_DIR.parent

DATA_DIR = BASE_DIR / "data" / "raw"
REPORTS_DIR = BASE_DIR / "reports" / "submissions"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

SRC_DIR = BASE_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from data.preprocess import clean_test_data, clean_train_data
from features.features import add_engineered_features, drop_highly_correlated_features


In [111]:
train_df_raw = pd.read_csv(DATA_DIR / "train.csv")
test_df_raw = pd.read_csv(DATA_DIR / "test.csv")
train_df = add_engineered_features(clean_train_data(train_df_raw))
test_df = add_engineered_features(clean_test_data(test_df_raw, train_df_raw))
train_df = drop_highly_correlated_features(train_df)
test_df = drop_highly_correlated_features(test_df)
TARGET_COL = "SalePrice"
X = train_df.drop(columns=[TARGET_COL]).copy()
y = train_df[TARGET_COL].copy()
y_log = np.log1p(y)
X_test = test_df.copy()


In [112]:
# Same split as ml_models_pipelines.ipynb, with log-transformed target
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y_log,
    test_size=0.2,
    random_state=42,
)
y_val_original = y.loc[y_val.index]
X_train.shape, X_val.shape


((1168, 82), (292, 82))

In [113]:
ordinal_categories = {
    "ExterQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    "ExterCond": ["Po", "Fa", "TA", "Gd", "Ex"],
    "BsmtQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    "BsmtCond": ["Po", "Fa", "TA", "Gd", "Ex"],
    "HeatingQC": ["Po", "Fa", "TA", "Gd", "Ex"],
    "KitchenQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    "FireplaceQu": ["Po", "Fa", "TA", "Gd", "Ex"],
    "GarageQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    "GarageCond": ["Po", "Fa", "TA", "Gd", "Ex"],
    "PoolQC": ["Fa", "TA", "Gd", "Ex"],
    "BsmtExposure": ["No", "Mn", "Av", "Gd"],
    "BsmtFinType1": ["Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "BsmtFinType2": ["Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "Functional": ["Sal", "Sev", "Maj2", "Maj1", "Mod", "Min2", "Min1", "Typ"],
    "GarageFinish": ["Unf", "RFn", "Fin"],
    "Fence": ["MnWw", "GdWo", "MnPrv", "GdPrv"],
    "LotShape": ["IR3", "IR2", "IR1", "Reg"],
    "Utilities": ["ELO", "NoSeWa", "NoSewr", "AllPub"],
    "LandSlope": ["Sev", "Mod", "Gtl"],
    "PavedDrive": ["N", "P", "Y"],
}

categorical_cols = X_train.select_dtypes(include=["object", "category", "string"]).columns.tolist()
ordinal_cols = [c for c in categorical_cols if c in ordinal_categories]
nominal_cols = [c for c in categorical_cols if c not in ordinal_cols]
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()


In [114]:
class SelectiveLogTransformer(BaseEstimator, TransformerMixin):
    """Apply log1p to selected numeric columns only."""
    def __init__(self, columns):
        self.columns = columns
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        for col in self.columns:
            if col in X.columns and pd.api.types.is_numeric_dtype(X[col]):
                min_val = X[col].min()
                if pd.notna(min_val) and min_val <= -1:
                    X[col] = X[col] + abs(min_val) + 1.0
                X[col] = np.log1p(X[col])
        return X
def remove_outliers_iqr_low_rate_columns(X_df, y_series, threshold_pct=10.0):
    """Drop rows that are outliers in columns whose outlier rate is < threshold_pct."""
    X_df = X_df.copy()
    y_series = y_series.copy()
    outlier_rates = []
    numeric = X_df.select_dtypes(include=[np.number]).columns.tolist()
    for col in numeric:
        q1 = X_df[col].quantile(0.25)
        q3 = X_df[col].quantile(0.75)
        iqr = q3 - q1
        if iqr == 0:
            continue
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        mask = (X_df[col] < lower) | (X_df[col] > upper)
        count = int(mask.sum())
        if count > 0:
            outlier_rates.append((col, 100 * count / len(X_df)))
    drop_candidate_cols = [col for col, pct in outlier_rates if pct < threshold_pct]
    rows_to_drop = pd.Series(False, index=X_df.index)
    for col in drop_candidate_cols:
        q1 = X_df[col].quantile(0.25)
        q3 = X_df[col].quantile(0.75)
        iqr = q3 - q1
        if iqr == 0:
            continue
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        rows_to_drop |= (X_df[col] < lower) | (X_df[col] > upper)
    X_clean = X_df.loc[~rows_to_drop].copy()
    y_clean = y_series.loc[~rows_to_drop].copy()
    return X_clean, y_clean, drop_candidate_cols, int(rows_to_drop.sum())
def build_preprocessor(scale_numeric=False):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    ordinal_categories_in_order = [ordinal_categories[col] for col in ordinal_cols]
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline(numeric_steps), numeric_cols),
            (
                "ord",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    (
                        "encoder",
                        OrdinalEncoder(
                            categories=ordinal_categories_in_order,
                            handle_unknown="use_encoded_value",
                            unknown_value=-1,
                        ),
                    ),
                ]),
                ordinal_cols,
            ),
            (
                "nom",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore")),
                ]),
                nominal_cols,
            ),
        ],
        remainder="drop",
    )
    return preprocessor
def evaluate_probabilistic(y_true_original, y_true_log, mu_log_pred, sigma_log_pred):
    sigma_safe = np.clip(np.asarray(sigma_log_pred), 1e-8, None)
    z_value = 1.959963984540054
    lower_log = mu_log_pred - z_value * sigma_safe
    upper_log = mu_log_pred + z_value * sigma_safe
    mu_pred = np.expm1(mu_log_pred)
    lower = np.expm1(lower_log)
    upper = np.expm1(upper_log)
    rmse = float(np.sqrt(mean_squared_error(y_true_original, mu_pred)))
    mse = float(mean_squared_error(y_true_original, mu_pred))
    r2 = float(r2_score(y_true_original, mu_pred))
    nll = float(-np.mean(norm.logpdf(y_true_log, loc=mu_log_pred, scale=sigma_safe)))
    coverage_95 = float(np.mean((y_true_original >= lower) & (y_true_original <= upper)))
    interval_widths = upper - lower
    width_95_median = float(np.median(interval_widths))
    width_95_mean = float(np.mean(interval_widths))
    return {
        "rmse": rmse,
        "mse": mse,
        "r2": r2,
        "nll": nll,
        "coverage_95": coverage_95,
        "width_95_median": width_95_median,
        "width_95_mean": width_95_mean,
    }


In [115]:
# Same outlier handling as linear family in ml_models_pipelines.ipynb
X_train_linear, y_train_linear, linear_drop_cols, dropped_rows = remove_outliers_iqr_low_rate_columns(
    X_train,
    y_train,
    threshold_pct=4.0,
)

print(f"Linear outlier columns used for row drop: {len(linear_drop_cols)}")
print(f"Rows dropped from training split: {dropped_rows}")
print(f"Train shape before: {X_train.shape} | after: {X_train_linear.shape}")


Linear outlier columns used for row drop: 16
Rows dropped from training split: 143
Train shape before: (1168, 82) | after: (1025, 82)


In [116]:
linear_log_columns = [
    col for col in ["OverallQual", "Age", "TotalSF", "TotalBath", "LotArea", "MasVnrArea"]
    if col in X_train.columns
]
prob_pipeline = Pipeline([
    ("log_transform", SelectiveLogTransformer(columns=linear_log_columns)),
    ("preprocessor", build_preprocessor(scale_numeric=True)),
    ("model", BayesianRidge()),
])
param_grid = {
    "model__alpha_1": [1e-6, 1e-5],
    "model__alpha_2": [1e-6, 1e-5],
    "model__lambda_1": [1e-6, 1e-5],
    "model__lambda_2": [1e-6, 1e-5],
}
gs = GridSearchCV(
    estimator=prob_pipeline,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
)
gs.fit(X_train_linear, y_train_linear)
best_model = gs.best_estimator_
cv_results_df = pd.DataFrame(gs.cv_results_)
trial_rows = []
for row in cv_results_df.itertuples(index=False):
    trial_params = row.params
    trial_estimator = prob_pipeline.set_params(**trial_params)
    trial_estimator.fit(X_train_linear, y_train_linear)
    val_mu_log, val_sigma_log = trial_estimator.predict(X_val, return_std=True)
    metrics = evaluate_probabilistic(
        y_true_original=y_val_original.to_numpy(),
        y_true_log=y_val.to_numpy(),
        mu_log_pred=val_mu_log,
        sigma_log_pred=val_sigma_log,
    )
    trial_rows.append(
        {
            "params": trial_params,
            "mean_cv_rmse": float(-row.mean_test_score),
            "rank_cv_rmse": int(row.rank_test_score),
            **metrics,
        }
    )
trials_df = pd.DataFrame(trial_rows)
# Set this to any available metric:
# mse, rmse, r2, nll, coverage_95, width_95_median, width_95_mean, mean_cv_rmse
SORT_BY = "mse"
ASCENDING_BY_METRIC = {
    "mse": True,
    "rmse": True,
    "r2": False,
    "nll": True,
    "coverage_95": False,
    "width_95_median": True,
    "width_95_mean": True,
    "mean_cv_rmse": True,
}
trials_df_sorted = trials_df.sort_values(
    by=SORT_BY,
    ascending=ASCENDING_BY_METRIC.get(SORT_BY, True),
).reset_index(drop=True)
trials_df_sorted.tail(3)


,params,mean_cv_rmse,rank_cv_rmse,rmse,mse,r2,nll,coverage_95,width_95_median,width_95_mean
13,"{'model__alpha_1': 1e-06, 'model__alpha_2': 1e...",0.107711,3,22704.328584,5.154865e+08,0.932795,-0.594472,0.976027,94785.112995,2.806390e+14
14,"{'model__alpha_1': 1e-05, 'model__alpha_2': 1e...",0.107711,2,22704.328684,5.154865e+08,0.932795,-0.594472,0.976027,94785.107498,2.806381e+14
15,"{'model__alpha_1': 1e-06, 'model__alpha_2': 1e...",0.107711,1,22704.328697,5.154865e+08,0.932795,-0.594472,0.976027,94785.107866,2.806380e+14


In [117]:
trials_df_sorted.columns

Index(['params', 'mean_cv_rmse', 'rank_cv_rmse', 'rmse', 'mse', 'r2', 'nll',
       'coverage_95', 'width_95_median', 'width_95_mean'],
      dtype='str')

In [118]:
# Fit on full data exactly like linear models workflow, then create submission
X_full_linear, y_full_linear, _, _ = remove_outliers_iqr_low_rate_columns(
    X,
    y_log,
    threshold_pct=4.0,
)
best_model.fit(X_full_linear, y_full_linear)
test_mu_log, test_sigma_log = best_model.predict(X_test, return_std=True)
test_mu = np.maximum(np.expm1(test_mu_log), 0)
submission = pd.DataFrame({"Id": test_df_raw["Id"], "SalePrice": test_mu})
out_path = REPORTS_DIR / "bayesian_ridge_probabilistic.csv"
submission.to_csv(out_path, index=False)
# Save best probabilistic model
BEST_MODELS_DIR = BASE_DIR / "reports" / "bestmodels"
BEST_MODELS_DIR.mkdir(parents=True, exist_ok=True)
model_out_path = BEST_MODELS_DIR / "bayesian_ridge_probabilistic.joblib"
joblib.dump(best_model, model_out_path)
pd.DataFrame(
    {
        "saved_submission": [str(out_path)],
        "saved_model": [str(model_out_path)],
        "pred_mean_min": [float(np.min(test_mu))],
        "pred_mean_max": [float(np.max(test_mu))],
        "pred_std_log_mean": [float(np.mean(test_sigma_log))],
    }
)


,saved_submission,saved_model,pred_mean_min,pred_mean_max,pred_std_log_mean
0,/home/amraas/projects/realestatecons/reports/s...,/home/amraas/projects/realestatecons/reports/b...,45370.957158,1.386093e+06,0.2117
